# 第 5 章 传播仿真 · 代码实战（Python）

> 本书章代码的 Python 实现，逐段中文注释；对应 MATLAB 版本见 `matlab/fourier/ch5/`。

## 习题

`ch5_exercises.py`

In [ ]:
# -*- coding: utf-8 -*-
"""书 5.7 节习题 5.1 / 5.3 / 5.5 / 5.6 的数值解答(解析部分见笔记页面)。

5.1 圆孔 + 临界采样:Fresnel 传播的参数设计(L1 = sqrt(M lam z));
5.3 功率守恒诊断:P = sum(sum(I)) dx^2,源面 vs 观察面;
5.5 高斯光束:z_R = pi w0^2/lam,临界距离 z = L^2/(M lam),
    数值传播 vs 式 (5.26) 解析辐照度,带宽判据检查;
5.6 split-step 等价性(式 5.15):10 x TF(2000 m) vs 单次 TF(20000 m)。
"""

import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "common"))
from base import coords
from ch5_prop import propTF, propIR

HERE = os.path.dirname(os.path.abspath(__file__))

# ================= 习题 5.1:圆孔 + 临界采样 =================
# w = 0.05 m, lam = 0.5 um, z = 1000 m, M = 500,临界采样。
w1, lam1, z1c, M1 = 0.05, 0.5e-6, 1000.0, 500
L1c = np.sqrt(M1 * lam1 * z1c)          # 临界采样:dx = L1/M = lam*z/L1 -> L1^2 = M*lam*z
dx1c = L1c / M1
FN = 1 / (2 * dx1c)                     # Nyquist 频率
B1 = 5.98 / w1                          # 有效带宽估计(式 5.10)
NF1 = w1 ** 2 / (lam1 * z1c)            # 菲涅耳数
print("========== 习题 5.1(圆孔 w=0.05 m, lam=0.5 um, z=1000 m, M=500) ==========")
print(f"(a) 临界采样:L1 = sqrt(M*lam*z) = {L1c:.3f} m,dx = {dx1c * 1e3:.2f} mm,"
      f"FN = 1/(2dx) = {FN:.0f} cyc/m")
print(f"(b) B1 = 5.98/w = {B1:.1f} cyc/m < FN = {FN:.0f} cyc/m ✓;"
      f"跨直径样本数 = 2w/dx = {2 * w1 / dx1c:.0f}")
print(f"(c) NF = w^2/(lam z) = {NF1:.1f};z_NF=1 为 {w1 ** 2 / lam1:.0f} m,"
      f"z = 1000 m 处 NF>1,处于菲涅耳区(圆孔是平滑孔径,式 4.28 判据 N_F≳1 即可用)")

L1 = L1c
x1, _ = coords(L1, M1)
X1, Y1 = np.meshgrid(x1, x1)
u1c = np.where(np.hypot(X1, Y1) < w1, 1.0, 0.0)      # 圆孔(半径 = 分母的约定,见 ch4 勘误)
I1c = np.abs(u1c) ** 2
print(f"(d) 圆孔功率 = pi*w^2 = {np.pi * w1 ** 2:.6f},离散 {np.sum(I1c) * dx1c ** 2:.6f}"
      f"(跨直径 {2 * w1 / dx1c:.0f} 样本的圆盘近似)")
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))
for ax, zt in zip(axes, [500.0, 1000.0, 2000.0]):
    I2 = np.abs(propTF(u1c, L1, lam1, zt)) ** 2      # 临界采样下 TF/IR 一致,画 TF 即可
    ax.imshow(I2 ** (1 / 3), extent=[x1[0], x1[-1], x1[0], x1[-1]],
              origin="lower", cmap="gray")
    lamz_over_L = lam1 * zt / L1                     # lam*z/L 相对 dx
    regime = "critical" if abs(lamz_over_L - dx1c) < 1e-12 else (
        f"TF over ({L1 / (lam1 * zt) * dx1c:.0f}x)" if lamz_over_L < dx1c
        else f"TF under ({lamz_over_L / dx1c:.0f}x)")
    ax.set_title(f"z = {zt:.0f} m, {regime}", fontsize=9)
    ax.set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
fig.suptitle("Ex 5.1(d): circular aperture Fresnel TF propagation, $I_2^{1/3}$", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(HERE, "out_ex5_1.png"), dpi=150, bbox_inches="tight")

# ================= 习题 5.3:功率守恒诊断 =================
# P = ∫∫ I dxdy ≈ sum(sum(I)) dx^2。对 sqr_beam 的 8 个案例算源/观察面功率。
Lq, Mq, lamq, wq = 0.5, 250, 0.5e-6, 0.051
dxq = Lq / Mq
xq, _ = coords(Lq, Mq)
Xq, Yq = np.meshgrid(xq, xq)
u1q = np.where((np.abs(Xq) < wq) & (np.abs(Yq) < wq), 1.0, 0.0)
P1 = np.sum(np.abs(u1q) ** 2) * dxq ** 2
print("\n========== 习题 5.3(功率守恒,源面功率 = %.6f W/m^2 x m^2) ==========" % P1)
print(f"{'z (m)':>8} | {'P_TF':>10} | {'P_TF/P1':>9} | {'P_IR':>10} | {'P_IR/P1':>9}")
for z in [1000.0, 2000.0, 4000.0, 20000.0]:
    u2tf = propTF(u1q, Lq, lamq, z)
    u2ir = propIR(u1q, Lq, lamq, z)
    Ptf = np.sum(np.abs(u2tf) ** 2) * dxq ** 2
    Pir = np.sum(np.abs(u2ir) ** 2) * dxq ** 2
    print(f"{z:>8.0f} | {Ptf:>10.6f} | {Ptf / P1:>9.5f} | {Pir:>10.6f} | {Pir / P1:>9.5f}")
print("(TF 各距离功率守恒到混叠精度;IR 长距离压掉 L/(2 lam z) 以外的源频谱,"
      "功率略降——z=20000 m 的 IR 案例损失最明显)")

# ================= 习题 5.5:高斯光束 =================
w0, lam5, A0 = 1e-3, 0.633e-6, 1.0                   # 式 (5.29) 参数
L5, M5 = 0.015, 250                                  # 边长 15 mm,250x250
dx5 = L5 / M5
zR = np.pi * w0 ** 2 / lam5                          # 式 (5.28):Rayleigh 范围
zcrit = L5 ** 2 / (M5 * lam5)                        # 临界采样距离(式 5.14)
x5, _ = coords(L5, M5)
X5, Y5 = np.meshgrid(x5, x5)
U0 = A0 * np.exp(-(X5 ** 2 + Y5 ** 2) / w0 ** 2)     # 式 (5.29)
print("\n========== 习题 5.5(高斯光束 w0=1 mm, lam=0.633 um, L=15 mm, M=250) ==========")
print(f"z_R = pi*w0^2/lam = {zR:.3f} m;临界采样距离 z = L^2/(M lam) = {zcrit:.3f} m")
# (b) 10 m 的源带宽判据:可用带宽 L/(2 lam z),高斯谱 exp(-pi^2 w0^2 f^2) 的
#     99% 功率带宽 B99 = erfinv(0.99)/(sqrt(2) pi w0)
from scipy.special import erfinv
B99 = erfinv(0.99) / (np.sqrt(2) * np.pi * w0)
avail = L5 / (2 * lam5 * 10.0)
print(f"(b) z=10 m 可用带宽 L/(2 lam z) = {avail:.0f} cyc/m;"
      f"高斯谱 99% 功率带宽 = {B99:.0f} cyc/m -> 满足 B1 < 可用带宽 ✓")
fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.4))
for ax, zt in zip(axes[:3], [1.0, 5.0, 10.0]):
    u2 = propTF(U0, L5, lam5, zt)
    I2 = np.abs(u2) ** 2
    wz = w0 * np.sqrt(1 + (zt / zR) ** 2)            # 式 (5.27)
    I_an = np.abs(A0) ** 2 * (w0 / wz) ** 2 * np.exp(-2 * (X5 ** 2 + Y5 ** 2) / wz ** 2)
    ax.plot(x5, I_an[M5 // 2, :], "--", lw=1.2, label="analytic (5.26)")
    ax.plot(x5, I2[M5 // 2, :], "-", lw=0.9, label="propTF")
    ax.set_title(f"z = {zt:.0f} m, w(z) = {wz * 1e3:.2f} mm", fontsize=9)
    ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    ax.set_xlabel("x (m)")
    err = np.abs(I2[M5 // 2, :] - I_an[M5 // 2, :]).max() / I_an[M5 // 2, :].max()
    print(f"(a) z={zt:>4.0f} m: 剖面最大相对偏差 = {err:.2e}(w(z) 解析 {wz * 1e3:.3f} mm)")
axes[3].imshow(np.abs(propTF(U0, L5, lam5, 10.0)) ** 2,
               extent=[x5[0], x5[-1], x5[0], x5[-1]], origin="lower", cmap="gray")
axes[3].set_title("z = 10 m image", fontsize=9)
axes[3].set_xlabel("x (m)")
fig.tight_layout()
fig.savefig(os.path.join(HERE, "out_ex5_5.png"), dpi=150)

# ================= 习题 5.6:split-step 等价性(式 5.15) =================
Ls, Ms, lams, ws = 0.5, 250, 0.5e-6, 0.051
dxs = Ls / Ms
xs, _ = coords(Ls, Ms)
Xs, Ys = np.meshgrid(xs, xs)
u1s = np.where((np.abs(Xs) < ws) & (np.abs(Ys) < ws), 1.0, 0.0)
u_single = propTF(u1s, Ls, lams, 20000.0)            # 单次 TF 20000 m
u_split = u1s.copy()
for _ in range(10):                                  # 10 次 TF,每步 2000 m(每步临界采样)
    u_split = propTF(u_split, Ls, lams, 2000.0)
dI = np.abs(np.abs(u_single) ** 2 - np.abs(u_split) ** 2).max()
print("\n========== 习题 5.6(split-step 等价性,总距离 20000 m) ==========")
print(f"max |I_single - I_split| = {dI:.3e}(浮点误差级 -> 式 (5.15) 成立:"
      f"H(z1)...H(zN) = H(z1+...+zN),split-step 不改变采样 regime)")
print("结论:想靠'拆成多个短距离、每步都满足判据'来救欠采样的长距离传播是行不通的"
      "(书 5.4.6 节);总传播距离才决定采样要求。")

print("\n已保存 out_ex5_1.png / out_ex5_5.png")


## 夫琅禾费衍射

`ch5_fraunhofer.py`

In [ ]:
# -*- coding: utf-8 -*-
"""书 5.5 节:sqr_beam_FF 夫琅禾费传播例子(复现图 5.7)。

改动相对 sqr_beam:方孔半宽 w = 0.011 m、用 propFF 传播、
显示时 nthroot(I2,3) 拉出旁瓣。菲涅耳数 NF = w^2/(lam z) = 0.12
——书原话"reasonable for the Fraunhofer approximation"。

观察面参数由式 (5.18) 决定:
  L2  = lam*z/dx1 = 0.5e-6*2000/0.002 = 0.5 m
  dx2 = lam*z/L1  = 0.5e-6*2000/0.5   = 2 mm
(本例恰好 dx1 = lam*z/L1 = 2 mm,即临界采样,L2 = L1。)

解析对照(式 5.20-5.22):
  FT{U1} = 4 w^2 sinc(2 w fX1) sinc(2 w fY1)
  I2(x2,y2) = (4 w^2/(lam z))^2 sinc^2(2 w x2/(lam z)) sinc^2(2 w y2/(lam z))
数值检查第一零点 x = lam*z/(2w) = 0.0455 m、第一旁瓣高度 ~0.045*I2peak。
MATLAB/Python 差异:nthroot(I2,3) 对应 I2**(1/3);np.sinc 与书表 1.2 的
sinc 同为 sin(pi u)/(pi u)。
"""

import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "common"))
from base import coords
from ch5_prop import propFF

# ---------- 源面(同 sqr_beam,只改 w) ----------
L1 = 0.5                                             # 边长 (m)
M = 250                                              # 样本数
dx1 = L1 / M                                         # 源面采样间隔 = 2 mm
lam = 0.5e-6                                         # 波长 (m)
k = 2 * np.pi / lam
w = 0.011                                            # 源方孔半宽 (m)(书改动行)
z = 2000.0                                           # 传播距离 (m)

x1, _ = coords(L1, M)
X1, Y1 = np.meshgrid(x1, x1)
u1 = np.where((np.abs(X1) < w) & (np.abs(Y1) < w), 1.0, 0.0)

# ---------- propFF 传播 ----------
u2, L2 = propFF(u1, L1, lam, z)
I2 = np.abs(u2) ** 2

dx2 = L2 / M
x2, _ = coords(L2, M)                                # 书改动行:x2 = -L2/2 : dx2 : L2/2-dx2

# ---------- 数值检查 ----------
peak_an = (4 * w ** 2 / (lam * z)) ** 2              # 式 (5.22) 峰值
null_an = lam * z / (2 * w)                          # sinc 第一零点
side_an = (np.sinc(1.5)) ** 2 * peak_an              # 第一旁瓣峰值(u = 1.5)
row = M // 2
prof = I2[row, :]
i_null = int(np.argmin(np.abs(x2 - null_an)))
i_side = int(np.argmax(prof[i_null + 5: i_null + 60])) + i_null + 5
print("========== 书 5.5 节 sqr_beam_FF 数值检查 ==========")
print(f"NF = w^2/(lam z) = {w ** 2 / (lam * z):.3f}(书:0.12)")
print(f"L2 = lam*z/dx1 = {L2:.3f} m;dx2 = lam*z/L1 = {lam * z / L1 * 1e3:.1f} mm(临界采样)")
print(f"I2 峰值:解析 (4w^2/(lam z))^2 = {peak_an:.4f},实测 {I2.max():.4f}")
print(f"sinc 第一零点 x = lam*z/(2w) = {null_an:.4f} m,实测最近样本 {x2[i_null]:.4f} m")
print(f"第一旁瓣:解析 {side_an:.4f}(0.045*I2peak),实测 {prof[i_side]:.4f}")
print(f"总功率:源面 {np.sum(np.abs(u1) ** 2) * dx1 ** 2:.6f} -> "
      f"观察面 {np.sum(I2) * dx2 ** 2:.6f}(FFT 归一化检查)")

# ---------- 图:复现图 5.7 ----------
fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.2))
axes[0].imshow(I2 ** (1 / 3), extent=[x2[0], x2[-1], x2[0], x2[-1]],
               origin="lower", cmap="gray")          # 书:nthroot(I2,3) 拉旁瓣
axes[0].set_xlabel("x (m)"); axes[0].set_ylabel("y (m)")
axes[0].set_title(f"z= {z:.0f} m  (a) $I_2^{{1/3}}$ pattern")
axes[1].plot(x2, prof)
axes[1].plot(x2[::5], (4 * w ** 2 / (lam * z)) ** 2
             * np.sinc(2 * w * x2[::5] / (lam * z)) ** 2, "o", ms=4,
             label="analytic Eq.(5.22)")             # 式 (5.22) 解析点
axes[1].set_xlabel("x (m)"); axes[1].set_ylabel("Irradiance")
axes[1].legend(fontsize=8)
axes[1].set_title(f"z= {z:.0f} m  (b) profile")
fig.tight_layout()
fig.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                         "out_fig5_7.png"), dpi=150)
print("\n已保存 out_fig5_7.png")


## 传播

`ch5_prop.py`

In [ ]:
# -*- coding: utf-8 -*-
"""书 5.1/5.2/5.5 节:三个 FFT 传播器(TF / IR / Fraunhofer)。

Python 侧的公共实现,供 ch5_sqr_beam.py / ch5_sampling.py /
ch5_fraunhofer.py / ch5_exercises.py 导入;与 MATLAB 侧的 propTF.m /
propIR.m / propFF.m 逐行对应(书 5.1、5.2、5.5 节)。

约定与书的记号一致:
  - 输入场 u1 按"居中排列"存放(零坐标样本在阵列中心),
    MATLAB 位置 (M/2+1, M/2+1) <-> numpy 位置 (M//2, M//2)。
  - 小写 u 表示空域场、大写 U 表示傅里叶域量(书 5.1 节末尾承认这种
    记号混用:"But what can you do?")。

MATLAB/Python 差异:
  - MATLAB fftshift/ifftshift 对偶数阵列完全相同,书全部取偶数 M,
    因此 Python 的 np.fft.fftshift / np.fft.ifftshift 可逐行照搬;
    若改成奇数阵列,两者不再等价(第 2 章坑与陷阱)。
  - MATLAB 的 1j 虚数单位与 Python 相同;MATLAB 数组索引 1-based,
    u2(M/2+1, :) 对应 Python 的 u2[M//2, :]。
  - 频率坐标轴用公共模块 base.coords(L, N) 生成(式 2.21/2.22),
    与书第 15 行 fx=-1/(2*dx):1/L:1/(2*dx)-1/L 完全一致。
"""

import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "common"))
from base import coords   # 公共积木(ch1 起):返回 (x, f) 两条居中坐标轴


# ===== 书 5.1 节:菲涅耳传递函数(TF)传播器 propTF =====

def propTF(u1, L, lam, z):
    """propagation - transfer function approach(书 5.1 节,式 5.1/5.2)。

    假设源面与观察面边长相同(L1 = L2 = L)、均匀采样。
    H = exp[jkz]·exp[-j pi lam z (fX^2+fY^2)] 中的整体相位 exp(jkz)
    不影响横向结构,按书第 18 行的做法直接丢掉。

    参数
    ----
    u1 : (M, M) 复数数组,居中排列的源面场
    L  : 源面/观察面边长 (m)
    lam: 波长 (m)
    z  : 传播距离 (m)

    返回
    ----
    u2 : (M, M) 复数数组,居中排列的观察面场
    """
    M, N = u1.shape                                   # 书 11 行:取阵列尺寸(只用 M)
    dx = L / M                                        # 书 12 行:采样间隔
    k = 2 * np.pi / lam                               # 书 13 行:波数
    _, fx = coords(L, M)                              # 书 15 行:频率坐标(居中)
    FX, FY = np.meshgrid(fx, fx)                      # 书 16 行(fy 与 fx 相同,省一行)
    H = np.exp(-1j * np.pi * lam * z * (FX ** 2 + FY ** 2))   # 书 18 行:传递函数(丢 e^{jkz})
    H = np.fft.fftshift(H)                            # 书 19 行:移位到 FFT 排列
    U1 = np.fft.fft2(np.fft.fftshift(u1))             # 书 20 行:源场移位 + FFT
    U2 = H * U1                                       # 书 21 行:逐点相乘
    u2 = np.fft.ifftshift(np.fft.ifft2(U2))           # 书 22 行:逆 FFT + 移回居中
    return u2


# ===== 书 5.2 节:菲涅耳脉冲响应(IR)传播器 propIR =====

def propIR(u1, L, lam, z):
    """propagation - impulse response approach(书 5.2 节,式 5.3/5.4)。

    与 propTF 解析上等价,但离散采样后行为不同(书 5.4 节的主题):
      - h 的 chirp 相位欠采样时,FFT 的周期延拓会在观察面产生周期副本;
      - h 过采样(Δx < λz/L)时,FFT 本身起到"窗口滤波"作用,
        压掉 L/(2 lam z) 以外的源频谱——长距离下反而更平滑。
    """
    M, N = u1.shape                                   # 书 11 行
    dx = L / M                                        # 书 12 行
    k = 2 * np.pi / lam                               # 书 13 行
    x, _ = coords(L, M)                               # 书 15 行:空间坐标(居中)
    X, Y = np.meshgrid(x, x)                          # 书 16 行
    h = 1 / (1j * lam * z) * np.exp(1j * k / (2 * z) * (X ** 2 + Y ** 2))  # 书 18 行:脉冲响应
    H = np.fft.fft2(np.fft.fftshift(h)) * dx ** 2     # 书 19 行:FFT 得传递函数,
    #                                                #   乘 dx^2 回到连续量纲——
    #                                                #   U1 与 U2 的 FFT/IFFT 定标互相抵消,
    #                                                #   只有 FFT{h} 这一步需要自己的定标(书 5.2 节注 b)
    U1 = np.fft.fft2(np.fft.fftshift(u1))             # 书 20 行
    U2 = H * U1                                       # 书 21 行
    u2 = np.fft.ifftshift(np.fft.ifft2(U2))           # 书 22 行
    return u2


# ===== 书 5.5 节:夫琅禾费传播器 propFF =====

def propFF(u1, L1, lam, z):
    """propagation - Fraunhofer pattern(书 5.5 节,式 5.16-5.19)。

    观察面边长与采样间隔由源面参数决定(式 5.18):
        L2  = lam*z/dx1,   dx2 = lam*z/L1
    所以函数额外返回 L2,省得主脚本再算一遍(书原话)。
    """
    M, N = u1.shape                                   # 书 11 行
    dx1 = L1 / M                                      # 书 12 行:源面采样间隔
    k = 2 * np.pi / lam                               # 书 13 行
    L2 = lam * z / dx1                                # 书 15 行:观察面边长
    dx2 = lam * z / L1                                # 书 16 行:观察面采样间隔
    x2, _ = coords(L2, M)                             # 书 17 行:观察面坐标(居中,与书式 5.19 一致)
    X2, Y2 = np.meshgrid(x2, x2)                      # 书 18 行
    c = 1 / (1j * lam * z) * np.exp(1j * k / (2 * z) * (X2 ** 2 + Y2 ** 2))  # 书 20 行:前导因子
    u2 = c * np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(u1))) * dx1 ** 2   # 书 21 行
    return u2, L2


## 采样

`ch5_sampling.py`

In [ ]:
# -*- coding: utf-8 -*-
"""书 5.4 节:菲涅耳传播的采样判据(复现图 5.4/5.5/5.6 + 表 5.2)。

内容:
  1) sqr_beam 源场在 z = 1000/2000/4000/20000 m 四个距离上分别用
     propTF / propIR 传播,画 x 轴辐照度剖面与图像(图 5.4/5.5)——
     观察"ticks"、"stair-step"、周期副本、假旁瓣四类 artifacts;
  2) 打印表 5.2(采样 regime、可用观察面尺寸、可用源带宽);
  3) 数值复核有效带宽估计 B1 = 5.98/w(式 5.10;ch 2.2 节式 2.10 的
     2D 径向 98% 谱功率方法);
  4) 与解析菲涅耳结果(式 5.11,方孔 + Fresnel 积分)对比:
     TF@1000、TF@2000、IR@20000(图 5.6,log 幅度 + unwrap 相位;
     解析曲线按书 5.4.4 节用 2500 点细网格评估)。
MATLAB/Python 差异:scipy.special.fresnel 返回 (S, C)(S 在前!),
MATLAB Symbolic Toolbox 用 mfun('FresnelC',x)/mfun('FresnelS',x)。
"""

import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import fresnel                    # 注意:返回 (S, C)

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "common"))
from base import coords
from ch5_prop import propTF, propIR

HERE = os.path.dirname(os.path.abspath(__file__))

# ---------- 源场(书 5.4.3 节式 5.9,参数同 sqr_beam) ----------
L = 0.5                                              # 边长 (m)
M = 250                                              # 样本数
dx = L / M                                           # 2 mm
lam = 0.5e-6                                         # 波长 (m)
w = 0.051                                            # 方孔半宽 (m)
D1 = 2 * w                                           # 源支撑 (m) = 0.102

x, _ = coords(L, M)
X, Y = np.meshgrid(x, x)
u1 = np.where((np.abs(X) < w) & (np.abs(Y) < w), 1.0, 0.0)   # 式 (5.9)

# ---------- 1) 四个距离的 TF / IR(图 5.4/5.5) ----------
distances = [1000.0, 2000.0, 4000.0, 20000.0]
results = {}
for z in distances:
    results[("TF", z)] = np.abs(propTF(u1, L, lam, z)) ** 2
    results[("IR", z)] = np.abs(propIR(u1, L, lam, z)) ** 2

fig, axes = plt.subplots(4, 2, figsize=(10.5, 9.5))
for i, z in enumerate(distances):
    axes[i, 0].plot(x, results[("TF", z)][M // 2, :])
    axes[i, 0].set_ylabel("irradiance")
    axes[i, 0].set_title(f"TF  z = {z:.0f} m", fontsize=9)
    axes[i, 1].plot(x, results[("IR", z)][M // 2, :])
    axes[i, 1].set_title(f"IR  z = {z:.0f} m", fontsize=9)
for ax in axes[-1, :]:
    ax.set_xlabel("x (m)")
for ax in axes.ravel():
    ax.set_ylim(-0.05, 2.1); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(HERE, "out_fig5_4.png"), dpi=150)

fig, axes = plt.subplots(4, 2, figsize=(9.0, 9.8))
for i, z in enumerate(distances):
    for j, key in enumerate(["TF", "IR"]):
        axes[i, j].imshow(results[(key, z)] ** (1 / 3),   # 1/3 次根拉出弱结构
                          extent=[x[0], x[-1], x[0], x[-1]],
                          origin="lower", cmap="gray")
        axes[i, j].set_title(f"{key}  z = {z:.0f} m", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(HERE, "out_fig5_5.png"), dpi=150)

# ---------- 2) 表 5.2 ----------
print("========== 表 5.2:sqr_beam 的采样 regime(dx = 2 mm, L = 0.5 m) ==========")
print(f"{'z (m)':>8} | {'lam*z/(L*dx)':>12} | TF(H)    | IR(h)    | "
      f"{'可用观察面宽 (m)':>18} | {'源带宽限制 (cyc/m)':>20}")
for z in distances:
    ratio = lam * z / (L * dx)                       # λz/(L·Δx):=1 为临界(书表 5.2 第 2 列)
    if ratio < 0.99:
        tf_s, ir_s = "over", "under"
        obs = f"D1+lam*z/dx = {D1 + lam * z / dx:.3f}"
        bw = f"1/(2dx) = {1 / (2 * dx):.0f}"
    elif ratio > 1.01:
        tf_s, ir_s = "under", "over"
        obs = f"L = {L}"
        bw = f"L/(2*lam*z) = {L / (2 * lam * z):.0f}"
    else:
        tf_s, ir_s = "critical", "critical"
        obs = f"L = {L}"
        bw = f"1/(2dx) = {1 / (2 * dx):.0f}"
    print(f"{z:>8.0f} | {ratio:>12.1f} | {tf_s:<8} | {ir_s:<8} | "
          f"{obs:>18} | {bw:>20}")
print("(书表 5.2:0.5/1/2/10 倍;可用带宽 250/250/125/25 cyc/m)")

# ---------- 3) B1 = 5.98/w 的数值复核(ch2 式 2.10 的 2D 径向 98% 方法) ----------
# |F|^2 = (4w^2)^2 sinc^2(2w fX) sinc^2(2w fY);令 u = 2w*rho(径向频率),
# 占比(B) = ∫_0^{2pi}∫_0^{U} sinc^2(u cos th) sinc^2(u sin th) u du dth / 总功率。
u_ax = np.linspace(1e-9, 40, 4001)
th = np.linspace(0, 2 * np.pi, 2881, endpoint=False)
Ug, Tg = np.meshgrid(u_ax, th, indexing="ij")
f_int = np.sinc(Ug * np.cos(Tg)) ** 2 * np.sinc(Ug * np.sin(Tg)) ** 2 * Ug
g_u = np.trapezoid(f_int, th, axis=1)                # 对 θ 积分
cum = np.concatenate([[0], np.cumsum((g_u[1:] + g_u[:-1]) / 2 * np.diff(u_ax))])
frac = cum / cum[-1]
u98 = float(np.interp(0.98, frac, u_ax))             # 98% 功率的无量纲半径
print("\n========== 式 (5.10) 复核(ch2 式 2.10:2D 径向 98% 谱功率) ==========")
print(f"98% 径向功率半径 u98 = 2w*B = {u98:.2f} -> B(98%) = {u98 / (2 * w):.1f} cyc/m")
print(f"书式 B1 = 5.98/w = {5.98 / w:.1f} cyc/m,对应径向功率占比 "
      f"{np.interp(2 * w * 5.98 / w, u_ax, frac):.3f}"
      f"(书值偏保守,多收了约 0.8% 的谱功率)")

# ---------- 4) 解析对比(式 5.11/5.13,图 5.6) ----------
def fresnel_1d_factor(x1d, w, lam, z):
    """式 (5.13) 的一维 Fresnel 积分因子 dC + j dS(不含前导 e^{jkz}/(2j))。

    xi1 = sqrt(2/(lam z))(-w-x), xi2 = sqrt(2/(lam z))(w-x);
    C/S 为 Fresnel 积分(式 5.12)。注意 scipy.special.fresnel 返回 (S, C)!
    """
    sq = np.sqrt(2.0 / (lam * z))
    xi1, xi2 = sq * (-w - x1d), sq * (w - x1d)
    S1, C1 = fresnel(xi1)                            # (S, C) —— S 在前
    S2, C2 = fresnel(xi2)
    return (C2 - C1) + 1j * (S2 - S1)


cases = [("TF", 1000.0), ("TF", 2000.0), ("IR", 20000.0)]
fig, axes = plt.subplots(3, 2, figsize=(10.5, 10.0))
for i, (key, z) in enumerate(cases):
    u2 = propTF(u1, L, lam, z) if key == "TF" else propIR(u1, L, lam, z)
    mag = np.abs(u2[M // 2, :])                      # 数值幅度剖面(250 点)
    pha = np.unwrap(np.angle(u2[M // 2, :]))         # 数值相位(unwrap)
    # 解析场(y = 0 剖面):U2 = e^{jkz}/(2j) * [dC+j dS]_x * [dC+j dS]_{y=0}
    U2_an = np.exp(1j * 2 * np.pi / lam * z) / (2j) \
        * fresnel_1d_factor(x, w, lam, z) * fresnel_1d_factor(np.array([0.0]), w, lam, z)[0]
    mag_an = np.abs(U2_an)
    pha_an250 = np.unwrap(np.angle(U2_an))           # 250 点解析(与数值同网格,定量比较用)
    # 书 5.4.4:解析式用 2500 点评估(检查仿真是否丢了空间细节)
    x_fine = np.linspace(-L / 2, L / 2 - L / 2500, 2500)
    U2_fine = np.exp(1j * 2 * np.pi / lam * z) / (2j) \
        * fresnel_1d_factor(x_fine, w, lam, z) * fresnel_1d_factor(np.array([0.0]), w, lam, z)[0]
    # unwrap 的整体 2*pi*N 分支任意:数值/解析曲线各自减去中心样本值再比较,
    # 否则两条曲线会差一个 2*pi 的整数倍常数(本次实测 TF@1000 差 8 圈 = 50.3 rad)
    pha = pha - pha[M // 2]
    pha_an250 = pha_an250 - pha_an250[M // 2]
    pha_an = np.unwrap(np.angle(U2_fine))
    pha_an = pha_an - pha_an[np.argmin(np.abs(x_fine))]

    axes[i, 0].semilogy(x_fine, np.abs(U2_fine), "--", lw=1.0, label="analytic (2500 pts)")
    axes[i, 0].semilogy(x, mag, "-", lw=0.9, label=f"{key} z={z:.0f} m (250 pts)")
    axes[i, 0].set_ylabel("Magnitude"); axes[i, 0].legend(fontsize=8)
    axes[i, 0].set_title(f"({chr(97 + 2 * i)}) {key} magnitude  z = {z:.0f} m", fontsize=9)
    axes[i, 1].plot(x_fine, pha_an, "--", lw=1.0, label="analytic (2500 pts)")
    axes[i, 1].plot(x, pha, "-", lw=0.9, label=f"{key} z={z:.0f} m")
    axes[i, 1].set_ylabel("Phase (rad)"); axes[i, 1].legend(fontsize=8)
    axes[i, 1].set_title(f"({chr(98 + 2 * i)}) {key} phase  z = {z:.0f} m", fontsize=9)
    # 定量偏差:|x| <= L/4 的剖面主体区(与同网格 250 点解析比较)
    core = np.abs(x) <= L / 4
    dphi = np.angle(np.exp(1j * (pha[core] - pha_an250[core])))
    print(f"{key} @ z={z:>6.0f} m: 主体区 max|dphi| = {np.abs(dphi).max():.4f} rad, "
          f"max|dMag|/Mag_an = {np.abs(mag[core] - mag_an[core]).max() / mag_an[core].max():.4f}")
for ax in axes[-1, :]:
    ax.set_xlabel("x (m)")
for ax in axes.ravel():
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(HERE, "out_fig5_6.png"), dpi=150)

print("\n已保存 out_fig5_4.png / out_fig5_5.png / out_fig5_6.png")


## 方形 · 光束

`ch5_sqr_beam.py`

In [ ]:
# -*- coding: utf-8 -*-
"""书 5.3 节:sqr_beam 方形光束传播例子(复现图 5.1/5.2/5.3)。

源面 0.5 m x 0.5 m(L1 = 0.5)、M = 250 采样、dx1 = 2 mm;
方孔半宽 w = 0.051 m(跨孔径 51 个样本)、单位振幅平面波背照明、
lam = 0.5 um、z = 2000 m、菲涅耳数 NF = w^2/(lam z) = 2.6
——书原话"reasonable for applying the Fresnel expression"。

同一脚本分别用 propTF 与 propIR 传播(书 5.3 节末尾:把 22 行换成
propIR 再跑一遍,结果应与图 5.2/5.3 完全一致),并做数值检查:
  1) TF 与 IR 的临界采样一致性(表 5.2:z = 2000 m 为 critical);
  2) 观察面峰值/中心辐照度,与解析 Fresnel 结果(式 5.11)对拍;
  3) unwrap 的作用说明(图 5.3(b) 的相位剖面)。
MATLAB/Python 差异:imagesc + axis xy 对应 imshow(origin='lower');
y = 0 的剖面 MATLAB 在 I2(M/2+1, :),numpy 在 I2[M//2, :]。
"""

import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)),
                                "..", "common"))
from base import coords                              # 公共积木(ch1 起)
from ch5_prop import propTF, propIR                  # 本章三个传播器(ch5_prop.py)

# ---------- 书 sqr_beam 第 1-7 行:源面网格 ----------
L1 = 0.5                                             # 边长 (m)
M = 250                                              # 样本数
dx1 = L1 / M                                         # 源面采样间隔 = 2 mm
x1, _ = coords(L1, M)                                # 书第 6 行:-L1/2 : dx1 : L1/2-dx1
y1 = x1                                              # 书第 7 行

# ---------- 书第 8-14 行:方孔源场 ----------
lam = 0.5e-6                                         # 波长 (m)
k = 2 * np.pi / lam                                  # 波数
w = 0.051                                            # 源方孔半宽 (m)
z = 2000.0                                           # 传播距离 (m)

X1, Y1 = np.meshgrid(x1, y1)                         # 书第 13 行
u1 = np.where((np.abs(X1) < w) & (np.abs(Y1) < w), 1.0, 0.0)   # 书第 14 行
#     rect(X1/(2w))*rect(Y1/(2w));矩形的边界值歧义对均匀网格无影响
I1 = np.abs(u1) ** 2                                 # 书第 15 行:源面辐照度

# ---------- 数值检查(书 5.3 节的口算) ----------
print("========== 书 5.3 节 sqr_beam 参数核对 ==========")
print(f"dx1 = L1/M = {dx1 * 1e3:.1f} mm(书:2 mm)")
print(f"跨孔径样本数 = 2w/dx1 = {2 * w / dx1:.0f}(书:51,对应 2.2 节的好表示)")
print(f"NF = w^2/(lam*z) = {w ** 2 / (lam * z):.2f}(书:2.6,'reasonable')")

# ---------- 书第 22 行:传播(TF 与 IR 各一次) ----------
u2_tf = propTF(u1, L1, lam, z)
u2_ir = propIR(u1, L1, lam, z)

I2_tf = np.abs(u2_tf) ** 2                           # 书第 26 行:观察面辐照度
I2_ir = np.abs(u2_ir) ** 2

row = M // 2                                         # y = 0 的行(numpy 0-based)
prof_tf = I2_tf[row, :]                              # 书第 35 行:I2(M/2+1, :)
prof_ir = I2_ir[row, :]

# ---------- 一致性 + 峰值检查 ----------
dmax = np.abs(I2_tf - I2_ir).max()
print("\n========== 数值检查 ==========")
print(f"max |I2_TF - I2_IR| / max(I2) = {dmax / I2_tf.max():.2e}"
      f"(临界采样下 TF 与 IR 应几乎逐位相同)")
print(f"I2 峰值 = {I2_tf.max():.4f}(图 5.2(b) 约 1.95,位于 x=±0.02;相长干涉超过源值 1)")
print(f"I2 中心 (0,0) = {I2_tf[row, row]:.4f}(中央峰;±0.01 处的谷 ≈0.89)")
print("剖面结构:中央峰(1.92)与侧峰(±0.02, 1.96)夹着 ±0.01 的谷(0.89)")
print(f"总功率: 源面 {np.sum(I1) * dx1 ** 2:.6f} -> 观察面 "
      f"{np.sum(I2_tf) * dx1 ** 2:.6f}(习题 5.3 的功率守恒诊断)")

# ---------- 图:复现图 5.1 / 5.2 / 5.3 ----------
fig1, ax = plt.subplots(figsize=(5.2, 4.4))          # 书 figure(1):图 5.1
ax.imshow(I1, extent=[x1[0], x1[-1], y1[0], y1[-1]],
          origin="lower", cmap="gray")               # imagesc + axis xy
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_title("z= 0 m")
fig1.tight_layout()
fig1.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                          "out_fig5_1.png"), dpi=150)

fig2, axes = plt.subplots(1, 2, figsize=(10.2, 4.2))  # 图 5.2
axes[0].imshow(I2_tf, extent=[x1[0], x1[-1], y1[0], y1[-1]],
               origin="lower", cmap="gray")
axes[0].set_xlabel("x (m)"); axes[0].set_ylabel("y (m)")
axes[0].set_title(f"z= {z:.0f} m  (a) irradiance pattern")
axes[1].plot(x1, prof_tf)
axes[1].set_xlabel("x (m)"); axes[1].set_ylabel("Irradiance")
axes[1].set_title(f"z= {z:.0f} m  (b) profile")
fig2.tight_layout()
fig2.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                          "out_fig5_2.png"), dpi=150)

fig3, axes = plt.subplots(1, 2, figsize=(10.2, 4.2))  # 图 5.3
axes[0].plot(x1, np.abs(u2_tf[row, :]))               # 书第 40 行:幅度剖面
axes[0].set_xlabel("x (m)"); axes[0].set_ylabel("Magnitude")
axes[0].set_title(f"z= {z:.0f} m  (a) magnitude profile")
axes[1].plot(x1, np.unwrap(np.angle(u2_tf[row, :])))  # 书第 45 行:unwrap 相位
axes[1].plot(x1, np.angle(u2_tf[row, :]), ":", lw=1)  # 去掉 unwrap 的效果(模 2pi)
axes[1].set_xlabel("x (m)"); axes[1].set_ylabel("Phase (rad)")
axes[1].legend(["unwrap(angle)", "angle (mod 2pi)"], fontsize=8)
axes[1].set_title(f"z= {z:.0f} m  (b) phase profile")
fig3.tight_layout()
fig3.savefig(os.path.join(os.path.dirname(os.path.abspath(__file__)),
                          "out_fig5_3.png"), dpi=150)

print("\n已保存 out_fig5_1.png / out_fig5_2.png / out_fig5_3.png")
